<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/09_three_source_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 — Phase 9: Three-Source Mixed Training

Phase 6 showed that mixing two forgery types (StyleGAN + face-swap) into
training dramatically improved generalization to those two types, but Phase 7
showed this benefit only partially transferred to a third, unrelated forgery
type (diffusion-generated faces). This phase asks the direct follow-up:
**does including a sample of the diffusion domain in training close that
remaining gap**, the way including face-swap closed the original gap?

**Important methodological note on data leakage:** the diffusion dataset was
used in its entirety as the *evaluation* set in Phases 7 and 8. If we now
train on those same images, evaluating on them again would be invalid
(the model would be tested on data it already saw). This notebook splits the
diffusion images into a training portion and a fresh, disjoint held-out
portion — evaluation later in this notebook uses **only** the held-out
portion, which is smaller than Phase 7/8's full evaluation set as a result.
This is called out explicitly wherever it affects comparability.

Compares three models on the held-out diffusion set, plus the StyleGAN and
face-swap test sets:
- Phase 2 (single-source: StyleGAN only)
- Phase 6 (two-source: StyleGAN + face-swap)
- Phase 9 (three-source: StyleGAN + face-swap + diffusion)

Requires `02_finetuning.ipynb` and `06_mixed_source_training.ipynb` to have
been run at least once (checkpoints in Drive).

Runtime → Change runtime type → GPU, then run cells top to bottom.

### Step 1 — Confirm you have a GPU

In [ ]:
!nvidia-smi

Sat Oct  3 17:14:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to / loaded from:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints will be saved to / loaded from: /content/drive/MyDrive/thesis_checkpoints


### Step 3 — Kaggle API access (via Colab Secrets)

Requires secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` already set up (🔑 icon,
left sidebar).

In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "YOUR_KAGGLE_API_TOKEN_HERE"

### Step 4 — Download all three datasets

In [ ]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip -d data_raw

!kaggle datasets download -d manjilkarki/deepfake-and-real-images
!unzip -q deepfake-and-real-images.zip -d data_raw_2

!kaggle datasets download -d bwandowando/faces-dataset-using-stable-diffusion-v14
!unzip -q faces-dataset-using-stable-diffusion-v14.zip -d data_diffusion
print("All three datasets downloaded.")

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:25<00:00, 159MB/s]

Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
100% 1.68G/1.68G [00:17<00:00, 103MB/s]

Dataset URL: https://www.kaggle.com/datasets/bwandowando/faces-dataset-using-stable-diffusion-v14
License(s): CC0-1.0
100% 1.10G/1.10G [00:12<00:00, 95.6MB/s]

All three datasets downloaded.


### Step 5 — Locate folders in all three datasets

In [ ]:
import os

def find_real_fake_folders(root):
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        lower = [d.lower() for d in dirnames]
        if "real" in lower and "fake" in lower:
            candidates.append(dirpath)
    return candidates

def find_split(root, keyword):
    candidates = find_real_fake_folders(root)
    matches = [x for x in candidates if keyword in x.lower()]
    return matches[0] if matches else (candidates[0] if candidates else None)

def find_class_dir(split_root, class_name):
    for d in os.listdir(split_root):
        if d.lower() == class_name:
            return os.path.join(split_root, d)
    raise FileNotFoundError(f"No '{class_name}' folder in {split_root}")

def find_all_images(root, exts=(".jpg", ".jpeg", ".png")):
    found = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if fname.lower().endswith(exts):
                found.append(os.path.join(dirpath, fname))
    return found

styleGAN_train = find_split("data_raw", "train")
styleGAN_test = find_split("data_raw", "test")
faceswap_train = find_split("data_raw_2", "train")
faceswap_test = find_split("data_raw_2", "test")
diffusion_images = find_all_images("data_diffusion")

print("StyleGAN train:", styleGAN_train)
print("StyleGAN test: ", styleGAN_test)
print("Face-swap train:", faceswap_train)
print("Face-swap test: ", faceswap_test)
print("Diffusion images found:", len(diffusion_images))

StyleGAN train: data_raw/real_vs_fake/real-vs-fake/train
StyleGAN test:  data_raw/real_vs_fake/real-vs-fake/test
Face-swap train: data_raw_2/Dataset/Train
Face-swap test:  data_raw_2/Dataset/Test
Diffusion images found: 2536


### Step 6 — Split diffusion images into a training portion and a fresh held-out evaluation portion

This is the critical leakage-prevention step. A fixed random seed makes this
split reproducible. Roughly 60% of the diffusion images go to training, 40%
are held out exclusively for evaluation later in this notebook.

In [ ]:
import random

rng = random.Random(2024)
shuffled_diffusion = diffusion_images[:]
rng.shuffle(shuffled_diffusion)

split_point = int(len(shuffled_diffusion) * 0.6)
diffusion_train_images = shuffled_diffusion[:split_point]
diffusion_heldout_images = shuffled_diffusion[split_point:]

print(f"Diffusion images for training: {len(diffusion_train_images)}")
print(f"Diffusion images held out for evaluation: {len(diffusion_heldout_images)}")
print("These two sets are disjoint by construction — no image appears in both.")

Diffusion images for training: 1521
Diffusion images held out for evaluation: 1015
These two sets are disjoint by construction — no image appears in both.


### Step 7 — Build the three-source mixed training set

Combines balanced samples from all three sources. Since the diffusion dataset
contains only fake images (no paired real photos of its own), the "real"
class is built from the StyleGAN and face-swap sources only — a reasonable
and common asymmetry when incorporating a fakes-only dataset, noted here
explicitly for methodological transparency.

Capped at 15,000 train / 2,000 val images per class per source for StyleGAN
and face-swap (matching Phase 6); the diffusion contribution is capped at
whatever is available in the training split from Step 6 (up to 1,500 train /
200 val).

In [ ]:
def build_threesource_split(dst_root,
                              styleGAN_root, faceswap_root, diffusion_train_pool,
                              n_train_per_source=15000, n_val_per_source=2000,
                              n_diffusion_train=1500, n_diffusion_val=200,
                              seed=42):
    rng = random.Random(seed)

    for cls in ["real", "fake"]:
        train_dst = os.path.join(dst_root, "train", cls)
        val_dst = os.path.join(dst_root, "val", cls)
        os.makedirs(train_dst, exist_ok=True)
        os.makedirs(val_dst, exist_ok=True)

        for source_root in [styleGAN_root, faceswap_root]:
            cls_dir = find_class_dir(source_root, cls)
            files = os.listdir(cls_dir)
            rng.shuffle(files)
            n_total = n_train_per_source + n_val_per_source
            chosen = files[:n_total]
            train_files, val_files = chosen[:n_train_per_source], chosen[n_train_per_source:n_total]
            source_tag = os.path.basename(os.path.normpath(source_root))
            for fname in train_files:
                src = os.path.abspath(os.path.join(cls_dir, fname))
                dst = os.path.join(train_dst, f"{source_tag}_{fname}")
                if not os.path.exists(dst):
                    os.symlink(src, dst)
            for fname in val_files:
                src = os.path.abspath(os.path.join(cls_dir, fname))
                dst = os.path.join(val_dst, f"{source_tag}_{fname}")
                if not os.path.exists(dst):
                    os.symlink(src, dst)

    # Diffusion contributes to the "fake" class only
    diffusion_shuffled = diffusion_train_pool[:]
    rng.shuffle(diffusion_shuffled)
    n_total_diff = n_diffusion_train + n_diffusion_val
    diff_chosen = diffusion_shuffled[:n_total_diff]
    diff_train, diff_val = diff_chosen[:n_diffusion_train], diff_chosen[n_diffusion_train:n_total_diff]

    train_fake_dst = os.path.join(dst_root, "train", "fake")
    val_fake_dst = os.path.join(dst_root, "val", "fake")
    for i, path in enumerate(diff_train):
        dst = os.path.join(train_fake_dst, f"diffusion_{i}_{os.path.basename(path)}")
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(path), dst)
    for i, path in enumerate(diff_val):
        dst = os.path.join(val_fake_dst, f"diffusion_{i}_{os.path.basename(path)}")
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(path), dst)

    return len(diff_train), len(diff_val)


n_diff_train_used, n_diff_val_used = build_threesource_split(
    "data_threesource", styleGAN_train, faceswap_train, diffusion_train_images
)

print("train/real:", len(os.listdir("data_threesource/train/real")))
print("train/fake:", len(os.listdir("data_threesource/train/fake")))
print("val/real:  ", len(os.listdir("data_threesource/val/real")))
print("val/fake:  ", len(os.listdir("data_threesource/val/fake")))
print(f"(of which {n_diff_train_used} train / {n_diff_val_used} val fake images are from the diffusion source)")

train/real: 30000
train/fake: 31500
val/real:   4000
val/fake:   4021
(of which 1500 train / 21 val fake images are from the diffusion source)


### Step 8 — Write the three-source training script

Identical recipe to Phase 6 (unfreeze `layer4` of ResNet18) — the only
variable changed is the training data, isolating the effect of adding a
third forgery type.

In [ ]:
%%writefile train_threesource.py
"""
Fine-tune ResNet18 (unfreeze layer4) on a three-source dataset
(StyleGAN + face-swap + diffusion combined).

Usage:
    python train_threesource.py --data_dir ./data_threesource --epochs 5 --batch_size 64 --lr 1e-4 --checkpoint_dir .
"""
import argparse
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm


def get_dataloaders(data_dir, batch_size, img_size=224):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ])
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize,
    ])
    train_set = datasets.ImageFolder(f"{data_dir}/train", transform=train_transform)
    val_set = datasets.ImageFolder(f"{data_dir}/val", transform=val_transform)
    print("Class mapping:", train_set.class_to_idx)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader, train_set.class_to_idx


def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, desc="train" if train else "val"):
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, default="./data_threesource")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--checkpoint_dir", type=str, default=".")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    train_loader, val_loader, class_to_idx = get_dataloaders(args.data_dir, args.batch_size)
    model = build_model().to(device)
    criterion = nn.CrossEntropyLoss()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable_params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {n_trainable:,} / {n_total:,}")

    optimizer = torch.optim.Adam(trainable_params, lr=args.lr)

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    save_path = os.path.join(args.checkpoint_dir, "best_model_threesource.pth")

    best_val_acc = 0.0
    for epoch in range(args.epochs):
        print(f"\nEpoch {epoch + 1}/{args.epochs}")
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
        print(f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({"model_state": model.state_dict(), "class_to_idx": class_to_idx}, save_path)
            print(f"Saved new best model to {save_path} (val_acc={val_acc:.4f})")

    print(f"\nDone. Best validation accuracy: {best_val_acc:.4f}")
    print(f"Final checkpoint is at: {save_path}")


if __name__ == "__main__":
    main()


Writing train_threesource.py


### Step 9 — Train on the three-source dataset

In [ ]:
!python train_threesource.py --data_dir ./data_threesource --epochs 5 --batch_size 64 --lr 1e-4 --checkpoint_dir "{CHECKPOINT_DIR}"

Using device: cuda
Class mapping: {'fake': 0, 'real': 1}
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 124MB/s]
Trainable parameters: 8,394,754 / 11,177,538

Epoch 1/5
train: 100% 961/961 [04:06<00:00,  3.90it/s]
val: 100% 126/126 [00:28<00:00,  4.48it/s]
train_loss=0.2314 train_acc=0.9008 val_loss=0.1264 val_acc=0.9481
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/best_model_threesource.pth (val_acc=0.9481)

Epoch 2/5
train: 100% 961/961 [03:47<00:00,  4.22it/s]
val: 100% 126/126 [00:23<00:00,  5.33it/s]
train_loss=0.1043 train_acc=0.9600 val_loss=0.1005 val_acc=0.9617
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/best_model_threesource.pth (val_acc=0.9617)

Epoch 3/5
train: 100% 961/961 [03:24<00:00,  4.71it/s]
val: 100% 126/126 [00:24<00:00,  5.11it/s]
train_loss=0.0672 train_acc=0.9749 val_loss=0.1063 val_acc=0.9620
Saved new 

### Step 10 — Build the three evaluation sets

StyleGAN and face-swap test sets as used throughout this project. The
diffusion evaluation set uses **only the held-out portion** from Step 6 —
never seen during this phase's training — paired with a fresh sample of
real photos from the StyleGAN test split (also never used in training, for
any phase).

In [ ]:
def build_eval_subset_imagefolder(source_root, dst_root, n_per_class, seed=99):
    rng = random.Random(seed)
    for cls in ["real", "fake"]:
        dst = os.path.join(dst_root, cls)
        os.makedirs(dst, exist_ok=True)
        cls_dir = find_class_dir(source_root, cls)
        files = os.listdir(cls_dir)
        rng.shuffle(files)
        for fname in files[:n_per_class]:
            src = os.path.abspath(os.path.join(cls_dir, fname))
            dst_path = os.path.join(dst, fname)
            if not os.path.exists(dst_path):
                os.symlink(src, dst_path)

build_eval_subset_imagefolder(styleGAN_test, "eval_styleGAN_v2", n_per_class=5000)
build_eval_subset_imagefolder(faceswap_test, "eval_faceswap_v2", n_per_class=5000)

# Diffusion held-out eval, paired with fresh StyleGAN test real images
real_dir = find_class_dir(styleGAN_test, "real")
real_images_pool = [os.path.join(real_dir, f) for f in os.listdir(real_dir)]

def build_eval_diffusion_heldout(dst_root, fake_paths, real_paths, seed=7):
    rng = random.Random(seed)
    fake_dst = os.path.join(dst_root, "fake")
    real_dst = os.path.join(dst_root, "real")
    os.makedirs(fake_dst, exist_ok=True)
    os.makedirs(real_dst, exist_ok=True)

    n = len(fake_paths)  # match real count to available held-out fakes
    real_sample = real_paths[:]
    rng.shuffle(real_sample)
    real_sample = real_sample[:n]

    for i, path in enumerate(fake_paths):
        dst = os.path.join(fake_dst, f"{i}_{os.path.basename(path)}")
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(path), dst)
    for i, path in enumerate(real_sample):
        dst = os.path.join(real_dst, f"{i}_{os.path.basename(path)}")
        if not os.path.exists(dst):
            os.symlink(os.path.abspath(path), dst)
    return n

n_eval = build_eval_diffusion_heldout("eval_diffusion_heldout", diffusion_heldout_images, real_images_pool)
print(f"Held-out diffusion eval set built: {n_eval} fake, {n_eval} real")
print("NOTE: this is a smaller, disjoint set from the one used in Phases 7-8 (leakage-free by construction).")

Held-out diffusion eval set built: 1015 fake, 1015 real
NOTE: this is a smaller, disjoint set from the one used in Phases 7-8 (leakage-free by construction).


### Step 11 — Evaluate Phase 2, Phase 6, and Phase 9 on all three domains

Phase 2 and Phase 6 are re-evaluated here on the *new* held-out diffusion set
(not the one used in Phases 7-8) so that the three-way comparison is
apples-to-apples — all three models evaluated on identical data.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), normalize])


def load_resnet(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()
    return model, checkpoint["class_to_idx"]


def evaluate(model, ckpt_class_to_idx, test_dir):
    real_idx = ckpt_class_to_idx["real"]
    test_set = datasets.ImageFolder(test_dir, transform=transform)
    loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2)
    remap = {idx: ckpt_class_to_idx[name.lower()] for name, idx in test_set.class_to_idx.items()}

    all_labels, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, real_idx].cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend([remap[l.item()] for l in labels])

    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    return acc, auc


models_to_eval = {
    "Phase 2 (single-source)": load_resnet(f"{CHECKPOINT_DIR}/best_model_finetuned.pth"),
    "Phase 6 (two-source)": load_resnet(f"{CHECKPOINT_DIR}/best_model_mixed.pth"),
    "Phase 9 (three-source)": load_resnet(f"{CHECKPOINT_DIR}/best_model_threesource.pth"),
}

test_sets = {
    "StyleGAN": "eval_styleGAN_v2",
    "Face-swap": "eval_faceswap_v2",
    "Diffusion (held-out)": "eval_diffusion_heldout",
}

print(f"{'Model':<25} {'Domain':<22} {'Accuracy':<12} {'AUC':<10}")
results = {}
for model_name, (model, cmap) in models_to_eval.items():
    for domain_name, test_dir in test_sets.items():
        acc, auc = evaluate(model, cmap, test_dir)
        results[(model_name, domain_name)] = (acc, auc)
        print(f"{model_name:<25} {domain_name:<22} {acc:<12.4f} {auc:<10.4f}")

print("\n--- Focus: Diffusion domain (the key question this phase answers) ---")
for model_name in models_to_eval:
    acc, auc = results[(model_name, "Diffusion (held-out)")]
    print(f"{model_name:<25} Accuracy={acc:.4f}  AUC={auc:.4f}")

Model                     Domain                 Accuracy     AUC       
Phase 2 (single-source)   StyleGAN               0.9870       0.9991    
Phase 2 (single-source)   Face-swap              0.4979       0.5227    
Phase 2 (single-source)   Diffusion (held-out)   0.4951       0.2849    
Phase 6 (two-source)      StyleGAN               0.9580       0.9951    
Phase 6 (two-source)      Face-swap              0.8254       0.9276    
Phase 6 (two-source)      Diffusion (held-out)   0.5365       0.5532    
Phase 9 (three-source)    StyleGAN               0.9601       0.9941    
Phase 9 (three-source)    Face-swap              0.8209       0.9132    
Phase 9 (three-source)    Diffusion (held-out)   0.9862       0.9987    

--- Focus: Diffusion domain (the key question this phase answers) ---
Phase 2 (single-source)   Accuracy=0.4951  AUC=0.2849
Phase 6 (two-source)      Accuracy=0.5365  AUC=0.5532
Phase 9 (three-source)    Accuracy=0.9862  AUC=0.9987


---
### What to save for this phase
- Diffusion train/held-out split sizes (Step 6)
- Training set composition and trainable parameter count (Steps 7, 9)
- The full 3-model x 3-domain table from Step 11, and especially the
  "Focus: Diffusion domain" comparison at the end

**How to interpret the result:**
- If Phase 9's Diffusion AUC is substantially higher than Phase 6's (which
  was ~0.55 on the *original*, larger diffusion eval set — note this run
  uses a different, smaller held-out set, so treat this as directionally
  comparable rather than exactly identical), that supports including a
  forgery type in training as the most direct way to gain robustness to it.
- Also check whether Phase 9's StyleGAN and Face-swap performance held up
  relative to Phase 6 — a good three-source model should not sacrifice
  either original domain much to gain the third.

Send me the full output and we'll add this as the next section of your report.